In [1]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [6]:
load_dotenv()
github_key = os.getenv("GITHUB_TOKEN")

headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_key}"
}

url = "https://api.github.com/repos/nodejs/node/pulls?state=all"
# response = requests.get(url, headers=headers)
# data = response.json()

In [7]:
all_results = []
cont = 0
while url:
    cont += 1
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        print(response.headers.get("Link", ""))
        data = response.json()
        
        all_results.extend(data)

        link_header = response.headers.get("Link", "")
        next_url = None
        for link in link_header.split(","):
            if 'rel="next"' in link:
                next_url = link[link.find("<")+1:link.find(">")]
                break

        url = next_url

        if cont == 30:
            url = None

        # 'don't get banned' check
        time.sleep(1)
        
    elif response.status_code == 202:
        print("Compiling data, try again shortly")
        break
    else:
        print(f"Error: {response.status_code}")
        break

<https://api.github.com/repositories/27193779/pulls?state=all&page=2>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1227>; rel="last"
<https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=3>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1227>; rel="last", <https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="first"
<https://api.github.com/repositories/27193779/pulls?state=all&page=2>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=4>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1227>; rel="last", <https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="first"
<https://api.github.com/repositories/27193779/pulls?state=all&page=3>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=5>; rel="nex

In [19]:
df = pd.json_normalize(
            all_results, 
            record_path=None, 
            meta=None, 
            errors='ignore'
        )
df.tail(100).to_json("../data/dataset/test/testo.json", orient="records")

In [ ]:
while True:
            print(f'Page: {page}..')
            response = requests.get(base_url + commits_url + f'?path={current_path}&per_page=100&page={page}', headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                print(f'Commits: {len(data)}')
                for commit in data:
                    author_info = commit.get('author')
                    if author_info and 'login' in author_info:
                        author_login = author_info['login']
                        all_authors[author_login] += 1

                    committer_info = commit.get('committer')
                    if committer_info and 'login' in committer_info:
                        committer_login = committer_info['login']
                        all_committers[committer_login] += 1


                # if less than 100 commits there's no more pages
                if len(data) < 100:
                    break 
                page += 1

            else:
                print(f"Error: {response.status_code}")
                break

In [36]:
cleaned_df = df.dropna(axis=1, how='all')
filtered_df = cleaned_df[["number", "state", "title", "body", "locked", "created_at", "updated_at", "closed_at", "merged_at", "assignees", "user.login", "labels", "author_association", "user.repos_url", "user.followers_url", "user.organizations_url", "user.starred_url", "user.type", "base.user.login", "base.user.followers_url", "base.user.starred_url", "base.repo.created_at", "base.repo.updated_at", "base.repo.pushed_at", "base.repo.size", "base.repo.releases_url", "base.repo.stargazers_count", "base.repo.watchers_count", "base.repo.language", "base.repo.has_issues", "base.repo.has_projects", "base.repo.has_downloads", "base.repo.has_projects", "base.repo.has_downloads", "base.repo.has_wiki", "base.repo.has_pages", "base.repo.has_discussions", "base.repo.forks_count"]]

In [27]:
url_columns = filtered_df.filter(regex='_url$')
url_columns

,user.repos_url,user.followers_url,user.organizations_url,user.starred_url,base.user.followers_url,base.user.starred_url,base.repo.releases_url
0,https://api.github.com/users/vagostep/repos,https://api.github.com/users/vagostep/followers,https://api.github.com/users/vagostep/orgs,https://api.github.com/users/vagostep/starred{...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
1,https://api.github.com/users/danmcd/repos,https://api.github.com/users/danmcd/followers,https://api.github.com/users/danmcd/orgs,https://api.github.com/users/danmcd/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
2,https://api.github.com/users/theoludwig/repos,https://api.github.com/users/theoludwig/followers,https://api.github.com/users/theoludwig/orgs,https://api.github.com/users/theoludwig/starre...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
3,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
4,https://api.github.com/users/LiviaMedeiros/repos,https://api.github.com/users/LiviaMedeiros/fol...,https://api.github.com/users/LiviaMedeiros/orgs,https://api.github.com/users/LiviaMedeiros/sta...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
...,...,...,...,...,...,...,...
895,https://api.github.com/users/joyeecheung/repos,https://api.github.com/users/joyeecheung/follo...,https://api.github.com/users/joyeecheung/orgs,https://api.github.com/users/joyeecheung/starr...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
896,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
897,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
898,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...


In [37]:
non_url_columns_df = filtered_df.drop(columns=[col for col in filtered_df.columns if col.endswith('_url')])

In [30]:
def fetch_url(url):
    try:
        print(url)
        response = requests.get(url)
        response.raise_for_status()
        return response.json()  # or response.text if it's not JSON
    except requests.RequestException as e:
        return f'Error: {e}'

# For each column, fetch and replace URLs with the data
count = 0
for col in url_columns.columns:
    print(f'col: {col}')
    count += 1
    url_columns[col] = url_columns[col].apply(fetch_url)
    time.sleep(1)
    if count == 30:
        break

col: user.repos_url
https://api.github.com/users/vagostep/repos
https://api.github.com/users/danmcd/repos
https://api.github.com/users/theoludwig/repos
https://api.github.com/users/StefanStojanovic/repos
https://api.github.com/users/LiviaMedeiros/repos
https://api.github.com/users/targos/repos
https://api.github.com/users/targos/repos
https://api.github.com/users/szegedi/repos
https://api.github.com/users/aduh95/repos
https://api.github.com/users/github-actions%5Bbot%5D/repos
https://api.github.com/users/puskin/repos
https://api.github.com/users/avivkeller/repos
https://api.github.com/users/mertcanaltin/repos
https://api.github.com/users/panva/repos
https://api.github.com/users/jazelly/repos
https://api.github.com/users/NishaGadave/repos
https://api.github.com/users/Skc-VitInProjects/repos
https://api.github.com/users/khardix/repos
https://api.github.com/users/lpinca/repos
https://api.github.com/users/panva/repos
https://api.github.com/users/jasnell/repos
https://api.github.com/users/j

KeyboardInterrupt: 

In [33]:
non_url_columns_df

,number,state,title,body,locked,created_at,updated_at,closed_at,merged_at,assignees,...,base.repo.language,base.repo.has_issues,base.repo.has_projects,base.repo.has_downloads,base.repo.has_projects,base.repo.has_downloads,base.repo.has_wiki,base.repo.has_pages,base.repo.has_discussions,base.repo.forks_count
0,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-05-08T21:50:39Z,2025-05-08T21:50:48Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
1,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: No connection adapters were found for '...,Error: Invalid URL 'False': No scheme supplied...,2025-05-08T19:21:08Z,2025-05-09T00:40:23Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
2,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-05-08T14:31:01Z,2025-05-09T01:18:19Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
3,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: No connection adapters were found for '...,Error: Invalid URL 'False': No scheme supplied...,2025-05-08T14:17:52Z,2025-05-08T20:52:10Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
4,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'None': No scheme supplied....,Error: Invalid URL 'False': No scheme supplied...,2025-05-08T11:37:08Z,2025-05-08T22:08:54Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: No connection adapters were found for '...,"Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-02-05T16:54:53Z,2025-02-11T10:58:59Z,2025-02-11T10:58:58Z,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
896,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-02-05T14:25:10Z,2025-02-07T14:33:55Z,2025-02-07T14:33:07Z,2025-02-07T14:33:07Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
897,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-02-05T14:21:43Z,2025-02-07T14:30:21Z,2025-02-07T14:24:04Z,2025-02-07T14:24:04Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
898,"Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...","Error: No connection adapters were found for ""...",Error: Invalid URL 'False': No scheme supplied...,2025-02-05T14:17:54Z,2025-02-07T14:23:56Z,2025-02-07T14:23:48Z,2025-02-07T14:23:48Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491


In [39]:
non_url_columns_df.to_csv("../data/dataset/pull_request_data.csv")
non_url_columns_df

,number,state,title,body,locked,created_at,updated_at,closed_at,merged_at,assignees,...,base.repo.language,base.repo.has_issues,base.repo.has_projects,base.repo.has_downloads,base.repo.has_projects,base.repo.has_downloads,base.repo.has_wiki,base.repo.has_pages,base.repo.has_discussions,base.repo.forks_count
0,58239,open,doc: spliting building steps for unix and macos,"Currently, there is an error when executing ex...",False,2025-05-08T21:50:39Z,2025-05-08T21:50:48Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
1,58237,open,deps: illumos madvise() pre-and-post-illumos#1...,"In illumos, madvise(3C) now takes `void *` for...",False,2025-05-08T19:21:08Z,2025-05-09T00:40:23Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
2,58236,open,"fs: glob is stable, so should not emit experim...","<!--\r\nBefore submitting a pull request, plea...",False,2025-05-08T14:31:01Z,2025-05-09T01:18:19Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
3,58235,open,"build,win: fix dll build",Fixes a linker error when building Node.js wit...,False,2025-05-08T14:17:52Z,2025-05-08T20:52:10Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
4,58233,open,test: add `Float16Array` to `common.getArrayBu...,None,False,2025-05-08T11:37:08Z,2025-05-08T22:08:54Z,None,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,56927,closed,[v20.x] backport unflagging of require(esm) to...,This backport includes the following PRs with ...,False,2025-02-05T16:54:53Z,2025-02-11T10:58:59Z,2025-02-11T10:58:58Z,None,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
896,56924,closed,doc: make MDN links to global classes more con...,"<!--\r\nBefore submitting a pull request, plea...",False,2025-02-05T14:25:10Z,2025-02-07T14:33:55Z,2025-02-07T14:33:07Z,2025-02-07T14:33:07Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
897,56923,closed,doc: make MDN links to global classes more con...,"<!--\r\nBefore submitting a pull request, plea...",False,2025-02-05T14:21:43Z,2025-02-07T14:30:21Z,2025-02-07T14:24:04Z,2025-02-07T14:24:04Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
898,56922,closed,doc: make MDN links to global classes more con...,"<!--\r\nBefore submitting a pull request, plea...",False,2025-02-05T14:17:54Z,2025-02-07T14:23:56Z,2025-02-07T14:23:48Z,2025-02-07T14:23:48Z,[],...,JavaScript,True,True,True,True,True,False,False,False,31491
